# Why LLM Inference Gets Fast and Then Runs Out of Memory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/kv_cache_llm_inference.ipynb)

This notebook accompanies the blog post: [Why LLM Inference Gets Fast and Then Runs Out of Memory](https://sesen.ai/blog/kv-cache-llm-inference-memory)

## What you will learn

1. **Why naive autoregressive generation is O(n²)** — measured empirically on GPT-2
2. **How the KV cache makes generation O(n)** — built from scratch in NumPy
3. **Why the cache costs so much memory** — the formula and real numbers for Llama-3.1 8B
4. **How quantisation solves the memory problem** — memory projections at 3-4 bits

## Setup

Install dependencies (only needed on Colab or fresh environments):

In [ ]:
# Uncomment on Colab
# !pip install transformers torch matplotlib numpy

In [ ]:
import time
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation, PillowWriter
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

---

## Part 1: The O(n²) Problem

Transformer generation is **autoregressive**: produce one token at a time, feed the entire sequence back in, produce the next token. Attention must compare every token to every other token at each step — O(n²) per step, O(n³) total naively.

Let's measure this directly on GPT-2 small.

In [ ]:
print("Loading GPT-2 small...")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
model.eval()
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def generate_naive(model, input_ids, n_tokens):
    """Recompute full attention at every step — O(n²) cost per step."""
    ids = input_ids.clone()
    with torch.no_grad():
        for _ in range(n_tokens):
            out = model(ids, use_cache=False)
            next_tok = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
            ids = torch.cat([ids, next_tok], dim=1)
    return ids


def generate_cached(model, input_ids, n_tokens):
    """Use past_key_values — O(n) computation per new token."""
    ids = input_ids.clone()
    past = None
    with torch.no_grad():
        for _ in range(n_tokens):
            out = model(
                ids if past is None else ids[:, -1:],
                past_key_values=past,
                use_cache=True,
            )
            past = out.past_key_values
            next_tok = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
            ids = torch.cat([ids, next_tok], dim=1)
    return ids

In [ ]:
prompt = "The history of artificial intelligence is a story of"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

seq_lengths = [50, 100, 200, 400]
n_runs = 2  # average over runs

naive_times = []
cached_times = []

for target_len in seq_lengths:
    # Naive timing
    times = []
    for _ in range(n_runs):
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        generate_naive(model, input_ids, target_len)
        if device == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    naive_times.append(np.mean(times))

    # Cached timing
    times = []
    for _ in range(n_runs):
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        generate_cached(model, input_ids, target_len)
        if device == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    cached_times.append(np.mean(times))

    print(f"{target_len:4d} tokens | naive {naive_times[-1]:.2f}s "
          f"| cached {cached_times[-1]:.2f}s "
          f"| {naive_times[-1]/cached_times[-1]:.1f}x faster")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(seq_lengths, naive_times, 'o-', color='#e05252', linewidth=2.5,
        markersize=8, label='Naive (recompute all attention)')
ax.plot(seq_lengths, cached_times, 's-', color='#4c9be8', linewidth=2.5,
        markersize=8, label='KV cache (incremental)')

ax.set_xlabel('Tokens generated', fontsize=13)
ax.set_ylabel('Wall-clock time (seconds)', fontsize=13)
ax.set_title('GPT-2 small: naive vs KV cache generation time', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_xticks(seq_lengths)

speedup = naive_times[-1] / cached_times[-1]
ax.annotate(f'{speedup:.1f}x faster',
            xy=(seq_lengths[-1], cached_times[-1]),
            xytext=(seq_lengths[-1] - 80, cached_times[-1] + (naive_times[-1] - cached_times[-1]) * 0.4),
            fontsize=11, color='#4c9be8',
            arrowprops=dict(arrowstyle='->', color='#4c9be8', lw=1.5))

plt.tight_layout()
plt.show()

---

## Part 2: Building a KV Cache from Scratch

Now let's build the cache mechanics from scratch in NumPy, using a minimal multi-head attention implementation.

### Q, K, V intuition

Each token is projected into three vectors:
- **Query (Q)**: "What am I looking for?" — the current token's question
- **Key (K)**: "What do I advertise?" — each past token's index card  
- **Value (V)**: "What do I actually contain?" — what to extract when matched

Only K and V need to be cached. The query for a past token is never needed again.

In [ ]:
def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)


class MultiHeadAttention:
    """Minimal multi-head attention with optional KV caching."""

    def __init__(self, d_model=64, n_heads=4, seed=0):
        rng = np.random.default_rng(seed)
        d = d_model
        self.n_heads = n_heads
        self.d_head = d // n_heads
        # Learned projections
        self.W_q = rng.standard_normal((d, d)) * 0.02
        self.W_k = rng.standard_normal((d, d)) * 0.02
        self.W_v = rng.standard_normal((d, d)) * 0.02
        self.W_o = rng.standard_normal((d, d)) * 0.02
        # Cache: list of arrays, one per head
        self.cache_k = [[] for _ in range(n_heads)]
        self.cache_v = [[] for _ in range(n_heads)]

    def reset_cache(self):
        self.cache_k = [[] for _ in range(self.n_heads)]
        self.cache_v = [[] for _ in range(self.n_heads)]

    def forward(self, x, use_cache=True):
        """
        x: (seq_len, d_model) — single token or full sequence
        Returns: (seq_len, d_model) output
        """
        seq_len, d = x.shape
        n_heads, d_head = self.n_heads, self.d_head

        # Project to Q, K, V
        Q = (x @ self.W_q).reshape(seq_len, n_heads, d_head)
        K = (x @ self.W_k).reshape(seq_len, n_heads, d_head)
        V = (x @ self.W_v).reshape(seq_len, n_heads, d_head)

        out_heads = []
        for h in range(n_heads):
            q = Q[:, h, :]      # (seq_len, d_head) — query for this head
            k_new = K[:, h, :]  # new keys to compute
            v_new = V[:, h, :]  # new values to compute

            if use_cache and len(self.cache_k[h]) > 0:
                # Retrieve all past K, V from cache and append new ones
                k_all = np.vstack(self.cache_k[h] + [k_new])
                v_all = np.vstack(self.cache_v[h] + [v_new])
            else:
                k_all = k_new
                v_all = v_new

            if use_cache:
                # Store the new token's K and V
                self.cache_k[h].append(k_new)
                self.cache_v[h].append(v_new)

            # Scaled dot-product attention
            scores = q @ k_all.T / np.sqrt(d_head)
            weights = softmax(scores)
            head_out = weights @ v_all
            out_heads.append(head_out)

        # Concatenate heads and project output
        out = np.concatenate(out_heads, axis=-1) @ self.W_o
        return out

In [ ]:
# Simulate token-by-token generation
mha = MultiHeadAttention(d_model=64, n_heads=4)
rng = np.random.default_rng(42)
tokens = rng.standard_normal((10, 64))  # 10 synthetic token embeddings

print("Token-by-token generation with KV cache:")
print(f"{'Step':>6} | {'Output shape':>14} | {'Total KV pairs cached':>22}")
print("-" * 50)

for step in range(10):
    token = tokens[step:step+1]  # feed one token at a time
    out = mha.forward(token, use_cache=True)
    total_cached = sum(len(mha.cache_k[h]) for h in range(mha.n_heads))
    print(f"Step {step+1:2d} | {str(out.shape):>14} | {total_cached:>22}")

In [ ]:
# Verify: cached and non-cached produce the same output for the full sequence
mha_full = MultiHeadAttention(d_model=64, n_heads=4)  # no caching

# Full sequence at once (no cache)
full_out = mha_full.forward(tokens, use_cache=False)

# Token by token (with cache) — already computed above in mha
# Rerun from scratch to compare last token output
mha_check = MultiHeadAttention(d_model=64, n_heads=4)
for step in range(10):
    cached_out = mha_check.forward(tokens[step:step+1], use_cache=True)

print("Output consistency check (last token):")
print(f"  Full-sequence output[-1]:  {full_out[-1, :4]}")
print(f"  Cached step-by-step[-1]:   {cached_out[0, :4]}")
print(f"  Max difference: {np.max(np.abs(full_out[-1] - cached_out[0])):.2e}")
print("  (small numerical difference expected from floating-point order)")

### Visualise the cache filling

Each step adds one blue (NEW) column. All previous columns turn green (retrieved from cache).

In [ ]:
tokens_text = ["The", "cat", "sat", "on", "the", "mat", "and", "then", "slept", "well"]
n_show = 10
n_layers = 3

fig, ax = plt.subplots(figsize=(10, 4.5))

# Show a mid-sequence state (token 6)
step = 5
ax.set_xlim(-0.5, n_show - 0.5)
ax.set_ylim(-0.5, n_layers - 0.5)
ax.set_xticks(range(n_show))
ax.set_xticklabels(tokens_text[:n_show], fontsize=10)
ax.set_yticks(range(n_layers))
ax.set_yticklabels([f'Layer {i+1}' for i in range(n_layers)], fontsize=11)
ax.set_xlabel('Token position', fontsize=11)

for x in range(n_show):
    for y in range(n_layers):
        color = '#f0f0f0'
        if x < step:
            color = '#a8d8a8'
        elif x == step:
            color = '#4c9be8'
        rect = mpatches.FancyBboxPatch((x - 0.45, y - 0.45), 0.9, 0.9,
                                       boxstyle="round,pad=0.05",
                                       facecolor=color, edgecolor='#888888',
                                       linewidth=0.8, alpha=0.85)
        ax.add_patch(rect)
        if x <= step:
            label = 'NEW' if x == step else 'KV'
            ax.text(x, y, label, ha='center', va='center',
                    fontsize=8, fontweight='bold',
                    color='white' if x == step else '#2d5a27')

new_patch = mpatches.Patch(color='#4c9be8', label='Computed this step')
cached_patch = mpatches.Patch(color='#a8d8a8', label='Retrieved from cache')
empty_patch = mpatches.Patch(facecolor='#f0f0f0', edgecolor='#888888', label='Not yet computed')
ax.legend(handles=[new_patch, cached_patch, empty_patch],
          loc='upper right', fontsize=9, framealpha=0.9)
ax.set_title(f'Token {step+1}/{n_show}: "{tokens_text[step]}" — KV Cache state', fontsize=13)

plt.tight_layout()
plt.show()

---

## Part 3: The Memory Problem

The KV cache trades compute for memory. Let's do the arithmetic for Llama-3.1 8B.

### The formula

```
Cache memory = 2 × L × H_kv × d_head × seq_len × bytes_per_element
```

- `2` = K and V
- `L` = number of layers  
- `H_kv` = number of KV heads (grouped-query attention)
- `d_head` = dimension per head
- `bytes_per_element` = 2 for BF16, 4 for FP32

In [ ]:
def kv_cache_gb(layers, kv_heads, head_dim, seq_len, bytes_per=2):
    """Calculate KV cache memory in GB."""
    total_bytes = 2 * layers * kv_heads * head_dim * seq_len * bytes_per
    return total_bytes / (1024 ** 3)


# Llama-3.1 8B parameters
L = 32        # transformer layers
H_kv = 8     # KV heads (grouped-query attention)
d_head = 128  # dimension per head
model_weights_gb = 16.0  # BF16 weights

print("Llama-3.1 8B KV cache memory at various context lengths (BF16):")
print(f"{'Context':>15} | {'Cache (GB)':>12} | {'vs Model Weights':>18}")
print("-" * 52)

for label, seq_len in [
    ("1K tokens", 1_024),
    ("4K tokens", 4_096),
    ("16K tokens", 16_384),
    ("32K tokens", 32_768),
    ("64K tokens", 65_536),
    ("128K tokens", 131_072),
]:
    cache = kv_cache_gb(L, H_kv, d_head, seq_len)
    pct = cache / model_weights_gb * 100
    print(f"{label:>15} | {cache:>12.1f} | {pct:>17.0f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

seq_lengths_bar = np.array([1_024, 4_096, 16_384, 32_768, 65_536, 131_072])
kv_cache_vals = np.array([kv_cache_gb(L, H_kv, d_head, s) for s in seq_lengths_bar])
seq_labels = ['1K', '4K', '16K', '32K', '64K', '128K']

# Left: bar chart
ax = axes[0]
colors = ['#4c9be8' if v < model_weights_gb else '#e05252' for v in kv_cache_vals]
bars = ax.bar(seq_labels, kv_cache_vals, color=colors, edgecolor='white', linewidth=0.5)
ax.axhline(model_weights_gb, color='#333333', linestyle='--', linewidth=2,
           label=f'Model weights ({model_weights_gb:.0f} GB)')
ax.set_xlabel('Context length', fontsize=12)
ax.set_ylabel('Memory (GB)', fontsize=12)
ax.set_title('KV cache size vs model weights\n(Llama-3.1 8B, BF16, batch=1)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, kv_cache_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{val:.1f} GB', ha='center', va='bottom', fontsize=9)

# Right: continuous growth with quantisation
ax2 = axes[1]
seq_cont = np.linspace(1000, 128000, 300)
kv_bf16 = np.array([kv_cache_gb(L, H_kv, d_head, s) for s in seq_cont])
kv_4bit = kv_bf16 * (4 / 16)
kv_3bit = kv_bf16 * (3 / 16)

ax2.plot(seq_cont / 1000, kv_bf16, color='#4c9be8', linewidth=2.5, label='KV cache (BF16)')
ax2.axhline(model_weights_gb, color='#333333', linestyle='--', linewidth=2,
            label=f'Model weights ({model_weights_gb:.0f} GB)')
ax2.plot(seq_cont / 1000, kv_4bit, color='#4c9be8', linewidth=1.5,
         linestyle=':', alpha=0.7, label='KV cache (4-bit)')
ax2.plot(seq_cont / 1000, kv_3bit, color='#4c9be8', linewidth=1.5,
         linestyle='-.', alpha=0.5, label='KV cache (3-bit)')
ax2.fill_between(seq_cont / 1000, kv_bf16, model_weights_gb,
                 where=(kv_bf16 > model_weights_gb),
                 alpha=0.15, color='#e05252', label='Cache exceeds weights')

ax2.set_xlabel('Context length (k tokens)', fontsize=12)
ax2.set_ylabel('Memory (GB)', fontsize=12)
ax2.set_title('KV cache memory growth\n(Llama-3.1 8B)', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, 128)

plt.tight_layout()
plt.show()

---

## Part 4: Quantisation as the Solution

Storing K and V vectors in 3-4 bits instead of 16 reduces cache memory by 4-5x with minimal quality loss.

The challenge is that transformer activations have **outliers** — a few coordinates carry most of the magnitude. Naive uniform quantisation on these vectors collapses the angular information that attention depends on.

The solution (TurboQuant) randomly rotates each vector before quantising. After rotation, the energy spreads evenly across all coordinates, making every quantisation bit count.

See the [TurboQuant blog post](https://sesen.ai/blog/turboquant-vector-quantization-random-rotations) for the full implementation.

In [ ]:
# Memory projections at different bit widths
print("Llama-3.1 8B KV cache at 128K context, different precisions:")
print(f"{'Precision':>12} | {'Cache (GB)':>12} | {'vs BF16':>12}")
print("-" * 42)

seq_len_128k = 131_072
bf16_cache = kv_cache_gb(L, H_kv, d_head, seq_len_128k, bytes_per=2)

for bits in [16, 8, 4, 3]:
    cache = kv_cache_gb(L, H_kv, d_head, seq_len_128k, bytes_per=bits/8)
    reduction = bf16_cache / cache
    print(f"{bits:>2}-bit (BF16):  {cache:>12.1f} | {reduction:>10.1f}x smaller")

In [ ]:
# Quick demo: why naive quantisation fails on outlier vectors
d = 128
rng = np.random.default_rng(0)

# Typical attention vector: one large outlier, the rest near zero
x = rng.standard_normal(d) * 0.01
x[0] = 1.0
x = x / np.linalg.norm(x)

# Naive uniform quantisation to 3 bits
n_levels = 8  # 2^3
x_min, x_max = x.min(), x.max()
grid = np.linspace(x_min, x_max, n_levels)
x_quantized = grid[np.argmin(np.abs(x[:, None] - grid), axis=1)]
mse_naive = np.mean((x - x_quantized) ** 2)

# After random rotation: energy spreads evenly
G = rng.standard_normal((d, d))
Q, R = np.linalg.qr(G)
Pi = Q * np.sign(np.diag(R))  # Haar-random rotation
x_rotated = Pi @ x

# Quantise rotated vector
x_rot_min, x_rot_max = x_rotated.min(), x_rotated.max()
grid_rot = np.linspace(x_rot_min, x_rot_max, n_levels)
x_rot_q = grid_rot[np.argmin(np.abs(x_rotated[:, None] - grid_rot), axis=1)]
x_reconstructed = Pi.T @ x_rot_q
mse_rotated = np.mean((x - x_reconstructed) ** 2)

print(f"Outlier vector (one large coordinate, rest near zero):")
print(f"  Naive 3-bit quantisation MSE:    {mse_naive:.6f}")
print(f"  After rotation, 3-bit quant MSE: {mse_rotated:.6f}")
print(f"  Improvement: {mse_naive / mse_rotated:.1f}x lower MSE")

# Visualise the coordinate distributions
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(x, bins=30, color='#e05252', alpha=0.7, edgecolor='white')
axes[0].set_title('Before rotation: one huge outlier', fontsize=12)
axes[0].set_xlabel('Coordinate value', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].axvline(0, color='black', linestyle='--', alpha=0.5)

axes[1].hist(x_rotated, bins=30, color='#4c9be8', alpha=0.7, edgecolor='white')
axes[1].set_title('After rotation: energy spread evenly', fontsize=12)
axes[1].set_xlabel('Coordinate value', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].axvline(0, color='black', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

---

## Exercises

1. **Extend the timing experiment**: Run GPT-2 medium (345M params) and compare the speedup ratios. Does the larger model benefit more or less from caching?

2. **Batch size scaling**: Modify `kv_cache_gb()` to add a `batch_size` parameter. Plot the cache memory for batch sizes 1, 4, 16, 32 at 32K context. At what batch size does the cache exceed the model weights for Llama-3.1 8B?

3. **Head dimension experiment**: The `MultiHeadAttention` class uses `d_head = d_model // n_heads`. What happens to the cached output quality if you vary `n_heads` while keeping `d_model` fixed? Try `n_heads = 1, 2, 4, 8` and compare the attention entropy (how spread out the attention weights are).

4. **Causal masking**: Add causal masking to the `MultiHeadAttention.forward()` method for the non-cached case (when processing a full sequence). Hint: create a lower-triangular mask and add a large negative value to masked positions before softmax.

5. **Grouped-query attention**: Modify `MultiHeadAttention` to implement grouped-query attention (GQA). With `n_heads = 8` and `n_kv_heads = 2`, each KV head should be shared by 4 query heads. Verify the output is the same shape and measure the cache size reduction.

6. **Memory calculator for your hardware**: What is the maximum context length you can run with your GPU's VRAM? Implement a function that takes `gpu_vram_gb`, `model_weights_gb`, `batch_size`, and the Llama architecture parameters, and returns the maximum feasible context length at a given bit-width.